# Association Rules on Segmentation — Revised Notebook

This revised version makes **minimal structural changes** to your original notebook and focuses only on three core updates:

1. **Separate 2-bin and 3-bin processing**
2. **Make the cluster-explanation workflow explicitly CAR (Class Association Rules)**
3. **Use the decision tree only as an optional auxiliary sanity check**

All other parts are kept as close as possible to the original workflow and output structure.


## Step 0. Imports

In [2]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

## Step 1. Parameters

This step keeps the same **centralized configuration style** as your original notebook.

The parameters you should pay most attention to are:

- `TWO_BIN_FEATURES`: variables explicitly assigned to 2-bin processing
- `MAX_ANT_LEN / MIN_SUPPORT_JOINT / MIN_CONFIDENCE / MIN_LIFT`: CAR rule thresholds
- `USE_DECISION_TREE`: whether to enable the optional decision-tree-based sanity check


In [3]:
# =========================
# Step 0) Parameters
# =========================

DATA_PATH = Path("customers_with_segments.csv")
TARGET = "Cluster"
FEATURES = [
    "MT_TTC_NET_total_spend",
    "CD_TICKET_UNIQUE_total_transactions",
    "avg_basket_value",
    "avg_items_per_basket",
    "product_diversity_score",
    "promo_percentage",
    "QTY_PROMO_STORE_store_promo_qty",
    "QTY_PROMO_NAT_nat_promo_qty",
    "ID_MAG_TIERS_stores_visited",
    "store_loyalty_score",
    "recency_days",
    "avg_days_between_visits",
]

# --- CAR thresholds ---
MAX_ANT_LEN = 2
MIN_SUPPORT_JOINT = 0.01
MIN_CONFIDENCE = 0.50
MIN_LIFT = 1.30
TOP_N_PER_CLUSTER = 10

# --- Small-cluster handling ---
USE_DYNAMIC_MIN_SUPPORT_BY_CLUSTER = True
MIN_CLUSTER_COVERAGE = 0.15  # A rule should cover at least 15% of a cluster
ABSOLUTE_MIN_JOINT_SUPPORT = 0.003  # global floor to avoid very tiny rules

# --- Binning strategy ---
# 2-bin features: explicitly zero-vs-positive (or low-vs-high if naturally binary)
TWO_BIN_FEATURES = {
    "promo_percentage",
    "QTY_PROMO_STORE_store_promo_qty",
    "QTY_PROMO_NAT_nat_promo_qty",
    "avg_days_between_visits",
    "CD_TICKET_UNIQUE_total_transactions",
}
THREE_BIN_FEATURES = [c for c in FEATURES if c not in TWO_BIN_FEATURES]

# --- Optional decision tree ---
USE_DECISION_TREE = True
TREE_MAX_DEPTH = 3
TREE_MIN_SAMPLES_LEAF = 0.02
TREE_RANDOM_STATE = 42

# --- Outputs ---
OUT_RULES_TOP = Path("cluster_explain_rules_top.csv")
OUT_PROFILE = Path("cluster_profile_report.csv")
OUT_BIN_META = Path("binning_metadata.csv")
OUT_TREE_RULES = Path("cluster_tree_rules.csv")
OUT_RULES_ALL = Path("cluster_explain_rules_all.csv")

# --- Validation (Step 5b) ---
USE_RULE_VALIDATION = True
VALIDATION_MODE = "holdout"   # "holdout" or "time_split"
DATE_COL = None               # e.g. "order_date"; if missing, code falls back to holdout
TEST_SIZE = 0.30
RANDOM_STATE = 42

# Stability thresholds on validation set
MIN_CONF_RATIO = 0.80
MIN_LIFT_RATIO = 0.80
MIN_SUPPORT_TEST = 0.005

# Fisher + BH(FDR)
USE_FISHER_FDR = True
FDR_ALPHA = 0.05

# --- Validation outputs ---
OUT_RULES_TRAIN = Path("cluster_explain_rules_train.csv")
OUT_RULES_VALIDATED = Path("cluster_explain_rules_validated.csv")
OUT_RULES_STABLE = Path("cluster_explain_rules_stable.csv")


## Step 2. Read and validate data

In [4]:
# =========================
# Step 1) Read and validate data
# =========================

def load_data(path: Path, extra_cols: List[str] | None = None) -> pd.DataFrame:
    extra_cols = [c for c in (extra_cols or []) if c]
    df = pd.read_csv(path)

    missing_cols = [c for c in FEATURES + [TARGET] + extra_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    keep_cols = FEATURES + [TARGET] + [c for c in extra_cols if c not in FEATURES + [TARGET]]
    df = df[keep_cols].copy()

    for c in FEATURES:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[TARGET] = pd.to_numeric(df[TARGET], errors="raise").astype(int)

    if df[FEATURES + [TARGET]].isna().any().any():
        na_info = df[FEATURES + [TARGET]].isna().sum()
        raise ValueError(
            "Found missing values after numeric conversion. Please clean input first.\n"
            f"{na_info[na_info > 0].to_string()}"
        )

    if extra_cols:
        for c in extra_cols:
            if "date" in c.lower() or "time" in c.lower():
                df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

# Load data

extra_cols = [DATE_COL] if USE_RULE_VALIDATION and VALIDATION_MODE == "time_split" and DATE_COL else []
df = load_data(DATA_PATH, extra_cols=extra_cols)
print('Shape:', df.shape)
print(df[TARGET].value_counts(normalize=True).sort_index().round(4))


Shape: (121112, 13)
Cluster
0    0.5171
1    0.0572
2    0.1446
3    0.2812
Name: proportion, dtype: float64


## Step 3. Explicit 2-bin / 3-bin strategy

This is one of the most important updates in the revised notebook:

- `bin_two_level`: handles variables explicitly assigned to 2-bin processing
- `bin_three_level`: keeps 3-bin processing for the remaining variables
- `build_binned_items`: converts each feature into a discretized item of the form `feature=label`


In [5]:
# =========================
# Step 2) Explicit 2-bin / 3-bin strategy
# =========================

@dataclass
class BinMeta:
    feature: str
    strategy: str
    labels: str
    threshold_1: float | None
    threshold_2: float | None
    note: str


def bin_two_level(s: pd.Series, feature: str) -> Tuple[pd.Series, BinMeta]:
    s = s.astype(float)

    # Most of these variables are effectively zero-inflated.
    # Make the semantics explicit instead of pretending they are low/mid/high.
    if (s == 0).mean() > 0.05:
        out = np.where(s <= 0, "zero", "positive")
        meta = BinMeta(
            feature=feature,
            strategy="2-bin_zero_vs_positive",
            labels="zero|positive",
            threshold_1=0.0,
            threshold_2=None,
            note="Explicitly separates 0 from >0 to avoid misleading qcut-based pseudo-3-bins.",
        )
    else:
        t = float(s.quantile(0.5))
        out = np.where(s <= t, "low", "high")
        meta = BinMeta(
            feature=feature,
            strategy="2-bin_median_split",
            labels="low|high",
            threshold_1=t,
            threshold_2=None,
            note="Median split used because feature is assigned to 2-bin treatment.",
        )
    return pd.Series(out, index=s.index, dtype="object"), meta


def bin_three_level(s: pd.Series, feature: str) -> Tuple[pd.Series, BinMeta]:
    s = s.astype(float)
    q1 = float(s.quantile(1 / 3))
    q2 = float(s.quantile(2 / 3))

    # If the variable collapses, degrade transparently instead of calling it 3-bin.
    if q1 == q2:
        out, meta2 = bin_two_level(s, feature)
        meta2.note += " Fallback triggered inside 3-bin request because q33 == q66."
        return out, meta2

    try:
        b = pd.qcut(s, q=3, labels=["low", "mid", "high"], duplicates="drop")
        cats = list(b.astype(str).unique())
        if len(set(cats)) == 3:
            meta = BinMeta(
                feature=feature,
                strategy="3-bin_qcut",
                labels="low|mid|high",
                threshold_1=q1,
                threshold_2=q2,
                note="Three-way quantile split.",
            )
            return pd.Series(b.astype(str), index=s.index), meta
    except Exception:
        pass

    # deterministic fallback
    out = np.where(s <= q1, "low", np.where(s <= q2, "mid", "high"))
    meta = BinMeta(
        feature=feature,
        strategy="3-bin_manual_quantile_split",
        labels="low|mid|high",
        threshold_1=q1,
        threshold_2=q2,
        note="Fallback manual split using q33 and q66.",
    )
    return pd.Series(out, index=s.index, dtype="object"), meta


def build_binned_items(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    binned = pd.DataFrame(index=df.index)
    meta_rows: List[BinMeta] = []

    for c in FEATURES:
        if c in TWO_BIN_FEATURES:
            bins, meta = bin_two_level(df[c], c)
        else:
            bins, meta = bin_three_level(df[c], c)

        binned[c] = bins.map(lambda x, c=c: f"{c}={x}")
        meta_rows.append(meta)

    binned["cluster_item"] = df[TARGET].map(lambda k: f"cluster={k}")
    meta_df = pd.DataFrame([m.__dict__ for m in meta_rows])
    return binned, meta_df

# Build binned items

binned, bin_meta = build_binned_items(df)
bin_meta.to_csv(OUT_BIN_META, index=False)
print(bin_meta)
print(f'Saved: {OUT_BIN_META.name}')

                                feature                strategy  \
0                MT_TTC_NET_total_spend              3-bin_qcut   
1   CD_TICKET_UNIQUE_total_transactions      2-bin_median_split   
2                      avg_basket_value              3-bin_qcut   
3                  avg_items_per_basket              3-bin_qcut   
4               product_diversity_score              3-bin_qcut   
5                      promo_percentage  2-bin_zero_vs_positive   
6       QTY_PROMO_STORE_store_promo_qty  2-bin_zero_vs_positive   
7           QTY_PROMO_NAT_nat_promo_qty  2-bin_zero_vs_positive   
8           ID_MAG_TIERS_stores_visited              3-bin_qcut   
9                   store_loyalty_score              3-bin_qcut   
10                         recency_days              3-bin_qcut   
11              avg_days_between_visits  2-bin_zero_vs_positive   

           labels  threshold_1  threshold_2  \
0    low|mid|high    16.550000    40.226667   
1        low|high     1.000000    

## Step 4. Encode row items to bitsets

In [6]:
# =========================
# Step 3) Encode row items to bitsets
# =========================

def encode_bitsets(binned: pd.DataFrame) -> Tuple[np.ndarray, Dict[str, int]]:
    items = sorted(pd.unique(binned[FEATURES].values.ravel()))
    if len(items) > 63:
        raise ValueError(
            f"Too many unique discrete items ({len(items)}). Current bitset encoding only supports <= 63."
        )

    item_to_bit = {item: i for i, item in enumerate(items)}
    bits = np.zeros(len(binned), dtype=np.uint64)
    one = np.uint64(1)
    for col in FEATURES:
        idx = binned[col].map(item_to_bit).to_numpy(dtype=np.uint64)
        bits |= (one << idx)
    return bits, item_to_bit

# Encode bitsets

bits, item_to_bit = encode_bitsets(binned)
clusters = sorted(df[TARGET].unique())
y = df[TARGET].to_numpy()
cluster_masks = {k: (y == k) for k in clusters}
p_cluster = {k: float(cluster_masks[k].mean()) for k in clusters}
print('n_items =', len(item_to_bit))

n_items = 31


## Step 5. Mine CAR rules

This step explicitly defines cluster explanation as **CAR (Class Association Rules)**:

- **Antecedent:** a combination of discretized feature labels
- **Consequent:** fixed as `cluster = k`

In other words, the general rule form is:

`A -> cluster = k`


In [7]:
# =========================
# Step 4) Mine CAR rules
# consequent is fixed as cluster=k
# =========================

def get_cluster_support_floor(cluster_prob: float) -> float:
    if not USE_DYNAMIC_MIN_SUPPORT_BY_CLUSTER:
        return MIN_SUPPORT_JOINT

    dynamic_floor = cluster_prob * MIN_CLUSTER_COVERAGE
    return max(ABSOLUTE_MIN_JOINT_SUPPORT, min(MIN_SUPPORT_JOINT, dynamic_floor))


def mine_car_rules(
    bits: np.ndarray,
    item_to_bit: Dict[str, int],
    clusters: List[int],
    cluster_masks: Dict[int, np.ndarray],
    p_cluster: Dict[int, float],
) -> pd.DataFrame:
    rows = []
    item_bits = sorted(
        [(item, (np.uint64(1) << np.uint64(bit))) for item, bit in item_to_bit.items()],
        key=lambda x: x[0],
    )

    for r in range(1, MAX_ANT_LEN + 1):
        for combo in combinations(item_bits, r):
            ant_items = [x[0] for x in combo]
            ant_bits = np.uint64(0)
            for _, b in combo:
                ant_bits |= b

            presence = (bits & ant_bits) == ant_bits
            supp_a = float(presence.mean())
            if supp_a < ABSOLUTE_MIN_JOINT_SUPPORT:
                continue

            for k in clusters:
                supp_joint = float((presence & cluster_masks[k]).mean())
                cluster_floor = get_cluster_support_floor(p_cluster[k])
                if supp_joint < cluster_floor:
                    continue

                conf = supp_joint / supp_a if supp_a > 0 else 0.0
                if conf < MIN_CONFIDENCE:
                    continue

                lift = conf / p_cluster[k] if p_cluster[k] > 0 else np.nan
                if lift < MIN_LIFT:
                    continue

                rows.append(
                    {
                        "cluster": k,
                        "antecedent": " & ".join(ant_items),
                        "consequent": f"cluster={k}",
                        "support": supp_joint,
                        "confidence": conf,
                        "lift": lift,
                        "support_antecedent": supp_a,
                        "cluster_base_rate": p_cluster[k],
                        "min_joint_support_used": cluster_floor,
                        "antecedent_len": r,
                        "rule_type": "CAR",
                    }
                )

    return pd.DataFrame(rows)

# Mine CAR rules

rules_df = mine_car_rules(bits, item_to_bit, clusters, cluster_masks, p_cluster)
print('rules found =', len(rules_df))
rules_df.head()

rules found = 228


,cluster,antecedent,consequent,support,confidence,lift,support_antecedent,cluster_base_rate,min_joint_support_used,antecedent_len,rule_type
0,2,ID_MAG_TIERS_stores_visited=high,cluster=2,0.072511,0.587307,4.062477,0.123464,0.144569,0.01,1,CAR
1,0,ID_MAG_TIERS_stores_visited=mid,cluster=0,0.164996,0.702267,1.358087,0.234948,0.517100,0.01,1,CAR
2,3,MT_TTC_NET_total_spend=low,cluster=3,0.194242,0.582691,2.072507,0.333353,0.281153,0.01,1,CAR
3,0,MT_TTC_NET_total_spend=mid,cluster=0,0.241446,0.724386,1.400862,0.333311,0.517100,0.01,1,CAR
4,3,avg_basket_value=low,cluster=3,0.182666,0.547748,1.948221,0.333485,0.281153,0.01,1,CAR


## Step 5b. Optional Rule Validation

This section adds a **validation layer** to the CAR workflow without changing the core mining logic.

Validation is performed with a **train/test split**:

- If `VALIDATION_MODE == "time_split"` and `DATE_COL` is available, the data is split in chronological order.
- Otherwise, the notebook uses a **stratified holdout split**.
- CAR rules are mined on the training set and then re-evaluated on the test set.
- The objective is to identify rules that remain stable, rather than rules that only look strong in a single sample.

In other words, this block is used to check whether high-confidence or high-lift rules are still meaningful on unseen data.


In [8]:
# =========================
# Step 5b) Optional rule validation (holdout / time split)
# =========================

def fit_bin_rules_from_train(df_train: pd.DataFrame) -> pd.DataFrame:
    """
    Fit binning thresholds on train only.
    Returns a bin_meta-like DataFrame that can be reused on test.
    """
    meta_rows = []
    for c in FEATURES:
        if c in TWO_BIN_FEATURES:
            _, meta = bin_two_level(df_train[c], c)
        else:
            _, meta = bin_three_level(df_train[c], c)
        meta_rows.append(meta.__dict__)
    return pd.DataFrame(meta_rows)


def apply_bins_from_meta(df_part: pd.DataFrame, bin_meta_ref: pd.DataFrame) -> pd.DataFrame:
    """
    Apply train-fitted binning thresholds to any split (train/test).
    """
    binned = pd.DataFrame(index=df_part.index)

    for _, row in bin_meta_ref.iterrows():
        feature = row["feature"]
        strategy = row["strategy"]
        t1 = row["threshold_1"]
        t2 = row["threshold_2"]
        s = df_part[feature].astype(float)

        if strategy == "2-bin_zero_vs_positive":
            labels = np.where(s <= 0, "zero", "positive")
        elif strategy == "2-bin_median_split":
            labels = np.where(s <= float(t1), "low", "high")
        else:
            labels = np.where(s <= float(t1), "low", np.where(s <= float(t2), "mid", "high"))

        binned[feature] = pd.Series(labels, index=s.index).map(lambda x, c=feature: f"{c}={x}")

    binned["cluster_item"] = df_part[TARGET].map(lambda k: f"cluster={k}")
    return binned


def split_for_validation(df_input: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    Time split if possible; otherwise stratified holdout.
    """
    if VALIDATION_MODE == "time_split" and DATE_COL and DATE_COL in df_input.columns:
        df_sorted = df_input.sort_values(DATE_COL).copy()
        cut = int(len(df_sorted) * (1 - TEST_SIZE))
        df_train_local = df_sorted.iloc[:cut].copy()
        df_test_local = df_sorted.iloc[cut:].copy()
        mode_used = f"time_split:{DATE_COL}"
    else:
        from sklearn.model_selection import train_test_split

        train_idx, test_idx = train_test_split(
            df_input.index,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=df_input[TARGET],
        )
        df_train_local = df_input.loc[train_idx].copy()
        df_test_local = df_input.loc[test_idx].copy()
        mode_used = "holdout"
    return df_train_local, df_test_local, mode_used


def parse_antecedent_items(antecedent: str) -> List[str]:
    return [x.strip() for x in antecedent.split("&") if x.strip()]


def build_item_sets_from_binned(binned_df: pd.DataFrame) -> List[set]:
    item_sets = []
    for _, row in binned_df[FEATURES].iterrows():
        item_sets.append(set(row.values.tolist()))
    return item_sets


def validate_car_rules_on_test(
    rules_train: pd.DataFrame,
    binned_test: pd.DataFrame,
    y_test: np.ndarray,
) -> pd.DataFrame:
    """
    Recompute support/confidence/lift for train-mined CAR rules on test set.
    """
    if rules_train.empty:
        return pd.DataFrame()

    test_item_sets = build_item_sets_from_binned(binned_test)
    n_test = len(test_item_sets)
    p_cluster_test = pd.Series(y_test).value_counts(normalize=True).to_dict()

    rows = []
    for _, r in rules_train.iterrows():
        ant_items = parse_antecedent_items(r["antecedent"])
        k = int(str(r["consequent"]).replace("cluster=", ""))

        match_mask = np.array([all(item in s for item in ant_items) for s in test_item_sets], dtype=bool)
        cluster_mask = (y_test == k)
        joint_mask = match_mask & cluster_mask

        support_test = float(joint_mask.mean())
        support_ant_test = float(match_mask.mean())
        confidence_test = support_test / support_ant_test if support_ant_test > 0 else 0.0
        base_k = float(p_cluster_test.get(k, 0.0))
        lift_test = confidence_test / base_k if base_k > 0 else np.nan

        conf_ratio = confidence_test / r["confidence"] if r["confidence"] > 0 else np.nan
        lift_ratio = lift_test / r["lift"] if r["lift"] > 0 else np.nan

        rows.append(
            {
                "cluster": int(r["cluster"]),
                "antecedent": r["antecedent"],
                "consequent": r["consequent"],
                "support_train": float(r["support"]),
                "confidence_train": float(r["confidence"]),
                "lift_train": float(r["lift"]),
                "support_test": support_test,
                "support_antecedent_test": support_ant_test,
                "confidence_test": confidence_test,
                "lift_test": lift_test,
                "conf_ratio": conf_ratio,
                "lift_ratio": lift_ratio,
            }
        )

    out = pd.DataFrame(rows)
    out["is_stable"] = (
        (out["support_test"] >= MIN_SUPPORT_TEST) &
        (out["conf_ratio"] >= MIN_CONF_RATIO) &
        (out["lift_ratio"] >= MIN_LIFT_RATIO)
    )
    return out


def add_fisher_bh_fdr(
    validated_rules: pd.DataFrame,
    binned_test: pd.DataFrame,
    y_test: np.ndarray,
) -> pd.DataFrame:
    """
    Add Fisher exact test p-values and BH(FDR) correction on test-set contingency tables.
    """
    if validated_rules.empty:
        return validated_rules.copy()

    try:
        from scipy.stats import fisher_exact
        from statsmodels.stats.multitest import multipletests
    except Exception as e:
        print(f"[WARN] scipy/statsmodels unavailable; skip Fisher + BH(FDR): {e}")
        out = validated_rules.copy()
        out["p_value"] = np.nan
        out["q_value"] = np.nan
        out["fdr_pass"] = False
        return out

    test_item_sets = build_item_sets_from_binned(binned_test)
    p_values = []

    for _, r in validated_rules.iterrows():
        ant_items = parse_antecedent_items(r["antecedent"])
        k = int(str(r["consequent"]).replace("cluster=", ""))

        match_mask = np.array([all(item in s for item in ant_items) for s in test_item_sets], dtype=bool)
        cluster_mask = (y_test == k)

        n11 = int(np.sum(match_mask & cluster_mask))
        n10 = int(np.sum(match_mask & (~cluster_mask)))
        n01 = int(np.sum((~match_mask) & cluster_mask))
        n00 = int(np.sum((~match_mask) & (~cluster_mask)))

        _, p = fisher_exact([[n11, n10], [n01, n00]], alternative="greater")
        p_values.append(p)

    out = validated_rules.copy()
    out["p_value"] = p_values

    reject, qvals, _, _ = multipletests(out["p_value"], alpha=FDR_ALPHA, method="fdr_bh")
    out["q_value"] = qvals
    out["fdr_pass"] = reject
    return out


# Run validation block

if USE_RULE_VALIDATION:
    df_train, df_test, validation_mode_used = split_for_validation(df)
    print(f"Validation mode used: {validation_mode_used}")
    print("Train shape:", df_train.shape, "| Test shape:", df_test.shape)

    # Fit bins on train only, then apply to both train and test
    bin_meta_train = fit_bin_rules_from_train(df_train)
    binned_train = apply_bins_from_meta(df_train, bin_meta_train)
    binned_test = apply_bins_from_meta(df_test, bin_meta_train)

    # Train-side CAR mining
    bits_train, item_to_bit_train = encode_bitsets(binned_train)
    clusters_train = sorted(df_train[TARGET].unique())
    y_train = df_train[TARGET].to_numpy()
    cluster_masks_train = {k: (y_train == k) for k in clusters_train}
    p_cluster_train = {k: float(cluster_masks_train[k].mean()) for k in clusters_train}

    rules_train = mine_car_rules(
        bits_train,
        item_to_bit_train,
        clusters_train,
        cluster_masks_train,
        p_cluster_train,
    )

    if not rules_train.empty:
        rules_train = rules_train.sort_values(
            ["cluster", "lift", "confidence", "support"],
            ascending=[True, False, False, False]
        ).reset_index(drop=True)
        rules_train.to_csv(OUT_RULES_TRAIN, index=False)
        print(f"Saved: {OUT_RULES_TRAIN.name}")
    else:
        print("No train-side CAR rules found under current thresholds.")

    # Validate on test
    rules_validated = validate_car_rules_on_test(
        rules_train=rules_train,
        binned_test=binned_test,
        y_test=df_test[TARGET].to_numpy(),
    )

    if USE_FISHER_FDR and not rules_validated.empty:
        rules_validated = add_fisher_bh_fdr(
            validated_rules=rules_validated,
            binned_test=binned_test,
            y_test=df_test[TARGET].to_numpy(),
        )
    elif not rules_validated.empty:
        rules_validated["p_value"] = np.nan
        rules_validated["q_value"] = np.nan
        rules_validated["fdr_pass"] = True

    if not rules_validated.empty:
        rules_validated.to_csv(OUT_RULES_VALIDATED, index=False)
        print(f"Saved: {OUT_RULES_VALIDATED.name}")

        stable_mask = rules_validated["is_stable"].fillna(False)
        if "fdr_pass" in rules_validated.columns:
            stable_mask = stable_mask & rules_validated["fdr_pass"].fillna(False)

        stable_rules = rules_validated.loc[stable_mask].copy()
        stable_rules = stable_rules.sort_values(
            ["cluster", "lift_test", "confidence_test", "support_test"],
            ascending=[True, False, False, False]
        ).reset_index(drop=True)
        stable_rules.to_csv(OUT_RULES_STABLE, index=False)
        print(f"Saved: {OUT_RULES_STABLE.name}")
        print("Validated rules:", len(rules_validated), "| Stable rules:", len(stable_rules))
    else:
        stable_rules = pd.DataFrame()
        print("Validation block produced no validated rules.")


Validation mode used: holdout
Train shape: (84778, 13) | Test shape: (36334, 13)
Saved: cluster_explain_rules_train.csv
Saved: cluster_explain_rules_validated.csv
Saved: cluster_explain_rules_stable.csv
Validated rules: 229 | Stable rules: 229


## Step 6. Select top rules and export

In [9]:
# =========================
# Step 5) Rule ranking and export
# =========================

def select_top_rules(rules_df: pd.DataFrame) -> pd.DataFrame:
    if rules_df.empty:
        return rules_df.copy()

    out = []
    for k in sorted(rules_df["cluster"].unique()):
        sub = (
            rules_df[rules_df["cluster"] == k]
            .sort_values(["lift", "confidence", "support"], ascending=[False, False, False])
            .head(TOP_N_PER_CLUSTER)
        )
        out.append(sub)
    return pd.concat(out, ignore_index=True)

# Select top rules

top_rules = select_top_rules(rules_df)
top_rules.to_csv(OUT_RULES_TOP, index=False)
rules_df.to_csv(OUT_RULES_ALL, index=False)
print(f'Saved: {OUT_RULES_ALL.name}')
print(f'Saved: {OUT_RULES_TOP.name}')
top_rules.head()

Saved: cluster_explain_rules_all.csv
Saved: cluster_explain_rules_top.csv


,cluster,antecedent,consequent,support,confidence,lift,support_antecedent,cluster_base_rate,min_joint_support_used,antecedent_len,rule_type
0,0,promo_percentage=zero & store_loyalty_score=mid,cluster=0,0.242346,0.968296,1.872552,0.250281,0.5171,0.01,2,CAR
1,0,MT_TTC_NET_total_spend=low & store_loyalty_sco...,cluster=0,0.104597,0.948701,1.834657,0.110253,0.5171,0.01,2,CAR
2,0,ID_MAG_TIERS_stores_visited=mid & MT_TTC_NET_t...,cluster=0,0.017752,0.945055,1.827606,0.018784,0.5171,0.01,2,CAR
3,0,MT_TTC_NET_total_spend=low & store_loyalty_sco...,cluster=0,0.018561,0.944538,1.826606,0.019651,0.5171,0.01,2,CAR
4,0,QTY_PROMO_NAT_nat_promo_qty=zero & store_loyal...,cluster=0,0.248349,0.943979,1.825525,0.263087,0.5171,0.01,2,CAR


## Step 7. Build human-readable profile report

In [10]:
# =========================
# Step 6) Human-readable profile text
# =========================

def item_to_text(item: str, bin_meta: pd.DataFrame) -> str:
    feature, label = item.split("=", 1)
    row = bin_meta.loc[bin_meta["feature"] == feature].iloc[0]
    t1, t2 = row["threshold_1"], row["threshold_2"]
    strategy = row["strategy"]

    if strategy == "2-bin_zero_vs_positive":
        mapping = {
            "zero": f"{feature} = 0",
            "positive": f"{feature} > 0",
        }
        return mapping.get(label, item)

    if strategy == "2-bin_median_split":
        mapping = {
            "low": f"{feature} <= {t1:.4g}",
            "high": f"{feature} > {t1:.4g}",
        }
        return mapping.get(label, item)

    mapping = {
        "low": f"{feature} <= {t1:.4g}",
        "mid": f"{t1:.4g} < {feature} <= {t2:.4g}",
        "high": f"{feature} > {t2:.4g}",
    }
    return mapping.get(label, item)


def antecedent_to_human(antecedent: str, bin_meta: pd.DataFrame) -> str:
    parts = [p.strip() for p in antecedent.split("&")]
    return " AND ".join(item_to_text(p, bin_meta) for p in parts)


def build_profile_report(top_rules: pd.DataFrame, bin_meta: pd.DataFrame) -> pd.DataFrame:
    if top_rules.empty:
        return top_rules.copy()

    df = top_rules.copy()
    df["p_cluster_est"] = df["confidence"] / df["lift"]
    df["coverage_in_cluster_est"] = df["support"] / df["p_cluster_est"]
    df["business_readable"] = df["antecedent"].apply(lambda x: antecedent_to_human(x, bin_meta))
    return df

# Build profile report

profile = build_profile_report(top_rules, bin_meta)
profile.to_csv(OUT_PROFILE, index=False)
print(f'Saved: {OUT_PROFILE.name}')
profile.head()

Saved: cluster_profile_report.csv


,cluster,antecedent,consequent,support,confidence,lift,support_antecedent,cluster_base_rate,min_joint_support_used,antecedent_len,rule_type,p_cluster_est,coverage_in_cluster_est,business_readable
0,0,promo_percentage=zero & store_loyalty_score=mid,cluster=0,0.242346,0.968296,1.872552,0.250281,0.5171,0.01,2,CAR,0.5171,0.468664,promo_percentage = 0 AND 0.3333 < store_loyalt...
1,0,MT_TTC_NET_total_spend=low & store_loyalty_sco...,cluster=0,0.104597,0.948701,1.834657,0.110253,0.5171,0.01,2,CAR,0.5171,0.202277,MT_TTC_NET_total_spend <= 16.55 AND 0.3333 < s...
2,0,ID_MAG_TIERS_stores_visited=mid & MT_TTC_NET_t...,cluster=0,0.017752,0.945055,1.827606,0.018784,0.5171,0.01,2,CAR,0.5171,0.034330,2 < ID_MAG_TIERS_stores_visited <= 3 AND MT_TT...
3,0,MT_TTC_NET_total_spend=low & store_loyalty_sco...,cluster=0,0.018561,0.944538,1.826606,0.019651,0.5171,0.01,2,CAR,0.5171,0.035895,MT_TTC_NET_total_spend <= 16.55 AND store_loya...
4,0,QTY_PROMO_NAT_nat_promo_qty=zero & store_loyal...,cluster=0,0.248349,0.943979,1.825525,0.263087,0.5171,0.01,2,CAR,0.5171,0.480272,QTY_PROMO_NAT_nat_promo_qty = 0 AND 0.3333 < s...


## Step 8. Optional decision tree sanity-check

This section is an **optional supporting module**, not the main workflow.

**Purpose:**

- Identify which variables are most important for separating clusters at the global level
- Perform a sanity check on the CAR results
- Help assess whether some high-lift rules are only local spikes rather than stable global patterns


In [11]:
# =========================
# Step 7) Optional decision tree for global sanity-check
# =========================

def run_decision_tree(df: pd.DataFrame) -> pd.DataFrame:
    if not USE_DECISION_TREE:
        return pd.DataFrame()

    try:
        from sklearn.tree import DecisionTreeClassifier, _tree
    except Exception as e:
        print(f"[WARN] scikit-learn not available, skip decision tree: {e}")
        return pd.DataFrame()

    X = df[FEATURES].copy()
    y = df[TARGET].copy()

    clf = DecisionTreeClassifier(
        max_depth=TREE_MAX_DEPTH,
        min_samples_leaf=TREE_MIN_SAMPLES_LEAF,
        random_state=TREE_RANDOM_STATE,
        class_weight="balanced",
    )
    clf.fit(X, y)

    tree_ = clf.tree_
    feature_names = FEATURES
    rows = []

    def recurse(node: int, conditions: List[str]) -> None:
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            f = feature_names[tree_.feature[node]]
            t = tree_.threshold[node]
            recurse(tree_.children_left[node], conditions + [f"{f} <= {t:.4g}"])
            recurse(tree_.children_right[node], conditions + [f"{f} > {t:.4g}"])
        else:
            values = tree_.value[node][0]
            pred_cluster = int(np.argmax(values))
            total = values.sum()
            purity = float(values[pred_cluster] / total) if total > 0 else np.nan
            rows.append(
                {
                    "cluster": pred_cluster,
                    "tree_rule": " AND ".join(conditions) if conditions else "ALL",
                    "node_samples": int(tree_.n_node_samples[node]),
                    "node_purity": purity,
                    "rule_type": "DecisionTree",
                }
            )

    recurse(0, [])
    return pd.DataFrame(rows).sort_values(["cluster", "node_purity", "node_samples"], ascending=[True, False, False])

# Run decision tree if enabled

tree_rules = run_decision_tree(df)
if not tree_rules.empty:
    tree_rules.to_csv(OUT_TREE_RULES, index=False)
    print(f'Saved: {OUT_TREE_RULES.name}')
tree_rules.head()

Saved: cluster_tree_rules.csv


,cluster,tree_rule,node_samples,node_purity,rule_type
0,0,promo_percentage <= 45.58 AND store_loyalty_sc...,57814,0.886675,DecisionTree
5,1,promo_percentage > 45.58 AND promo_percentage ...,3512,0.999768,DecisionTree
6,1,promo_percentage > 45.58 AND promo_percentage ...,2423,0.965820,DecisionTree
4,1,promo_percentage > 45.58 AND promo_percentage ...,2441,0.754862,DecisionTree
1,2,promo_percentage <= 45.58 AND store_loyalty_sc...,21128,0.887936,DecisionTree


## Step 9. Keep original product-rule summary block

In [12]:
# =========================
# Step 8) Keep original product-rule summary block almost unchanged
# =========================

def parse_segment_id(path: Path) -> int:
    m = re.search(r"association_rules_segment_(\d+)", path.name)
    if not m:
        raise ValueError(f"Cannot parse segment id from filename: {path.name}")
    return int(m.group(1))


def summarize_product_rules() -> None:
    files = sorted(Path(".").glob("association_rules_segment_*.csv"))
    if not files:
        print("Didn't find association_rules_segment_*.csv")
        return

    seg_reports = []
    for f in files:
        seg = parse_segment_id(f)
        df = pd.read_csv(f)
        top = df.sort_values(["lift", "confidence", "support"], ascending=False).head(10).copy()
        top.insert(0, "cluster", seg)
        top.insert(1, "source_file", f.name)
        seg_reports.append(top)

    seg_report = pd.concat(seg_reports, ignore_index=True)
    seg_report.to_csv("cluster_product_rules_top.csv", index=False)
    print("Saved: cluster_product_rules_top.csv")

# Summarize existing product rule files

summarize_product_rules()

Saved: cluster_product_rules_top.csv


## Step 10. Save config snapshot

In [13]:

config_snapshot = {
    "DATA_PATH": str(DATA_PATH),
    "TARGET": TARGET,
    "FEATURES": FEATURES,
    "TWO_BIN_FEATURES": sorted(TWO_BIN_FEATURES),
    "THREE_BIN_FEATURES": THREE_BIN_FEATURES,
    "MAX_ANT_LEN": MAX_ANT_LEN,
    "MIN_SUPPORT_JOINT": MIN_SUPPORT_JOINT,
    "MIN_CONFIDENCE": MIN_CONFIDENCE,
    "MIN_LIFT": MIN_LIFT,
    "TOP_N_PER_CLUSTER": TOP_N_PER_CLUSTER,
    "USE_DYNAMIC_MIN_SUPPORT_BY_CLUSTER": USE_DYNAMIC_MIN_SUPPORT_BY_CLUSTER,
    "MIN_CLUSTER_COVERAGE": MIN_CLUSTER_COVERAGE,
    "ABSOLUTE_MIN_JOINT_SUPPORT": ABSOLUTE_MIN_JOINT_SUPPORT,
    "USE_DECISION_TREE": USE_DECISION_TREE,
    "TREE_MAX_DEPTH": TREE_MAX_DEPTH,
    "TREE_MIN_SAMPLES_LEAF": TREE_MIN_SAMPLES_LEAF,
    "TREE_RANDOM_STATE": TREE_RANDOM_STATE,
    "USE_RULE_VALIDATION": USE_RULE_VALIDATION,
    "VALIDATION_MODE": VALIDATION_MODE,
    "DATE_COL": DATE_COL,
    "TEST_SIZE": TEST_SIZE,
    "RANDOM_STATE": RANDOM_STATE,
    "MIN_CONF_RATIO": MIN_CONF_RATIO,
    "MIN_LIFT_RATIO": MIN_LIFT_RATIO,
    "MIN_SUPPORT_TEST": MIN_SUPPORT_TEST,
    "USE_FISHER_FDR": USE_FISHER_FDR,
    "FDR_ALPHA": FDR_ALPHA,
}
Path("run_config_snapshot.json").write_text(json.dumps(config_snapshot, indent=2), encoding="utf-8")
print("Saved: run_config_snapshot.json")


Saved: run_config_snapshot.json


## Optional: one-click run all

If you would like to run the whole notebook in a script-like way, you can execute the following cell directly.


In [14]:
# =========================
# Step 9) Main pipeline
# =========================

def main() -> None:
    print("[1/10] Loading data...")
    extra_cols = [DATE_COL] if USE_RULE_VALIDATION and VALIDATION_MODE == "time_split" and DATE_COL else []
    df = load_data(DATA_PATH, extra_cols=extra_cols)
    print("Shape:", df.shape)
    print("Cluster distribution:")
    print(df[TARGET].value_counts(normalize=True).sort_index().round(4).to_string())

    print("\n[2/10] Building explicit 2-bin / 3-bin items...")
    binned, bin_meta = build_binned_items(df)
    bin_meta.to_csv(OUT_BIN_META, index=False)
    print(f"Saved: {OUT_BIN_META.name}")

    print("\n[3/10] Encoding bitsets...")
    bits, item_to_bit = encode_bitsets(binned)
    clusters = sorted(df[TARGET].unique())
    y = df[TARGET].to_numpy()
    cluster_masks = {k: (y == k) for k in clusters}
    p_cluster = {k: float(cluster_masks[k].mean()) for k in clusters}

    print("\n[4/10] Mining CAR rules...")
    rules_df = mine_car_rules(bits, item_to_bit, clusters, cluster_masks, p_cluster)
    if rules_df.empty:
        print("No CAR rules found under current thresholds.")
    else:
        rules_df = rules_df.sort_values(["cluster", "lift", "confidence", "support"], ascending=[True, False, False, False])
        rules_df.to_csv(OUT_RULES_ALL, index=False)
        print(f"Saved: {OUT_RULES_ALL.name}")

    if USE_RULE_VALIDATION:
        print("\n[5/10] Running holdout / time-split validation...")
        df_train, df_test, validation_mode_used = split_for_validation(df)
        print(f"Validation mode used: {validation_mode_used}")
        print("Train shape:", df_train.shape, "| Test shape:", df_test.shape)

        bin_meta_train = fit_bin_rules_from_train(df_train)
        binned_train = apply_bins_from_meta(df_train, bin_meta_train)
        binned_test = apply_bins_from_meta(df_test, bin_meta_train)

        bits_train, item_to_bit_train = encode_bitsets(binned_train)
        clusters_train = sorted(df_train[TARGET].unique())
        y_train = df_train[TARGET].to_numpy()
        cluster_masks_train = {k: (y_train == k) for k in clusters_train}
        p_cluster_train = {k: float(cluster_masks_train[k].mean()) for k in clusters_train}

        rules_train = mine_car_rules(
            bits_train, item_to_bit_train, clusters_train, cluster_masks_train, p_cluster_train
        )

        if not rules_train.empty:
            rules_train = rules_train.sort_values(
                ["cluster", "lift", "confidence", "support"],
                ascending=[True, False, False, False],
            ).reset_index(drop=True)
            rules_train.to_csv(OUT_RULES_TRAIN, index=False)
            print(f"Saved: {OUT_RULES_TRAIN.name}")

            rules_validated = validate_car_rules_on_test(
                rules_train=rules_train,
                binned_test=binned_test,
                y_test=df_test[TARGET].to_numpy(),
            )

            if USE_FISHER_FDR and not rules_validated.empty:
                rules_validated = add_fisher_bh_fdr(
                    validated_rules=rules_validated,
                    binned_test=binned_test,
                    y_test=df_test[TARGET].to_numpy(),
                )
            elif not rules_validated.empty:
                rules_validated["p_value"] = np.nan
                rules_validated["q_value"] = np.nan
                rules_validated["fdr_pass"] = True

            if not rules_validated.empty:
                rules_validated.to_csv(OUT_RULES_VALIDATED, index=False)
                print(f"Saved: {OUT_RULES_VALIDATED.name}")

                stable_mask = rules_validated["is_stable"].fillna(False)
                if "fdr_pass" in rules_validated.columns:
                    stable_mask = stable_mask & rules_validated["fdr_pass"].fillna(False)

                stable_rules = rules_validated.loc[stable_mask].copy()
                stable_rules = stable_rules.sort_values(
                    ["cluster", "lift_test", "confidence_test", "support_test"],
                    ascending=[True, False, False, False],
                ).reset_index(drop=True)
                stable_rules.to_csv(OUT_RULES_STABLE, index=False)
                print(f"Saved: {OUT_RULES_STABLE.name}")
            else:
                print("Validation block produced no validated rules.")
        else:
            print("No train-side CAR rules found for validation.")
    else:
        print("\n[5/10] Validation disabled.")

    print("\n[6/10] Selecting top rules per cluster...")
    top_rules = select_top_rules(rules_df)
    top_rules.to_csv(OUT_RULES_TOP, index=False)
    print(f"Saved: {OUT_RULES_TOP.name}")

    print("\n[7/10] Building profile report...")
    profile = build_profile_report(top_rules, bin_meta)
    profile.to_csv(OUT_PROFILE, index=False)
    print(f"Saved: {OUT_PROFILE.name}")

    if USE_DECISION_TREE:
        print("\n[8/10] Running optional decision tree sanity-check...")
        tree_rules = run_decision_tree(df)
        if not tree_rules.empty:
            tree_rules.to_csv(OUT_TREE_RULES, index=False)
            print(f"Saved: {OUT_TREE_RULES.name}")
        else:
            print("Decision tree block skipped or produced no output.")
    else:
        print("\n[8/10] Decision tree sanity-check disabled.")

    print("\n[9/10] Summarizing existing product rules...")
    summarize_product_rules()

    print("\n[10/10] Saving config snapshot...")
    Path("run_config_snapshot.json").write_text(json.dumps(config_snapshot, indent=2), encoding="utf-8")
    print("Saved: run_config_snapshot.json")

print("Notebook helpers loaded. Run main() for end-to-end execution.")


Notebook helpers loaded. Run main() for end-to-end execution.
